[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aecelaya/MIST/blob/main/examples/mist_planning_calibration.ipynb)

# Calibrating MIST's Patch and Batch Size Planning

MIST chooses the patch and batch size at the start of training from an estimate of
each network's GPU memory. The estimate's constants were fitted to one run of one
network in FP32, and its BF16 and inference constants are educated guesses. This
notebook measures real GPU memory so they can be replaced with data.

**Run it once per GPU: G4, L4, A100, and T4.** Choose the GPU under *Runtime > Change
runtime type*, then *Runtime > Run all*. Each run saves a CSV named after the GPU
and downloads it. No data is needed; inputs are random.

What it does for each network and precision:

- **Training:** runs real steps (forward, backward, AdamW) at several patch sizes
  and batch sizes, recording peak memory.
- **Inference:** runs forward passes without gradients at batches of 1, 2, and 4.

Each measurement records two peaks. cuDNN can use far more memory the first time it
sees a shape, so `peak_*` covers every step and `steady_*` leaves out the first.

BF16 runs only on GPUs with hardware support, as in real MIST runs. The A100, L4, and
G4 will measure both FP32 and BF16; the T4 predates Ampere, so it measures FP32 only.
It takes a while, and a run that gets disconnected only loses its remaining rows,
since results are saved after every measurement.

## Setup

In [ ]:
!pip install -q "git+https://github.com/aecelaya/MIST.git"

In [ ]:
import gc
import os
import time
import warnings

import numpy as np
import pandas as pd
import torch

from mist.models.model_registry import get_model_from_registry
from mist.training import patch_planning as pp
from mist.training.planning_constants import PlanningConstants as constants
from mist.utils import hardware

warnings.filterwarnings("ignore")
hardware.tune_gpu_math()  # TF32 convolutions and cuDNN benchmark, as in MIST's training.

GIB = 1024**3

# Set MIST_CALIBRATION_DRY_RUN to test this notebook's code without a GPU. Memory
# then reads as 0, so the results mean nothing.
DRY_RUN = bool(os.environ.get("MIST_CALIBRATION_DRY_RUN"))
if not (torch.cuda.is_available() or DRY_RUN):
    raise RuntimeError("No GPU found. Choose one under Runtime > Change runtime type.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEMORY = float(torch.cuda.get_device_properties(0).total_memory)
    torch.zeros(1, device=DEVICE)  # Start the CUDA context so it can be measured.
    free, total = torch.cuda.mem_get_info(0)
    CONTEXT_BYTES = float(total - free)
else:
    GPU_NAME, GPU_MEMORY, CONTEXT_BYTES = "dry-run", 4.0 * GIB, 0.0

PRECISIONS = ["fp32"] + (["bf16"] if hardware.bf16_supported() else [])
OUTPUT_CSV = f"mist_calibration_{GPU_NAME.replace(' ', '_').replace('/', '-')}.csv"

print(f"GPU: {GPU_NAME}, {GPU_MEMORY / GIB:.1f} GiB, context {CONTEXT_BYTES / GIB:.2f} GiB")
print(f"Precisions to measure: {PRECISIONS}")
print(f"Results go to {OUTPUT_CSV}")

## What to measure

One network per family, plus the heavier variants and the anisotropic case most
likely to behave differently. The patch sizes come from the planner itself, so they
cover the sizes real runs would use: it's asked to plan for fractions of this GPU's
memory, and each is run at batch sizes of 2 and 4.

In [ ]:
NETWORKS = [
    "nnunet-plain",
    "nnunet-res",
    "nnunet-pocket",
    "resenc",
    "fmgnet",
    "wnet",
    "mednext-small",
    "mednext-large",
    "swinunetr-small",
    "primus-medium",
    "primusv2-small",
]
SPACINGS = {
    "isotropic": [1.0, 1.0, 1.0],
    "anisotropic": [0.6, 0.6, 3.0],
}
ANISOTROPIC_NETWORKS = {"nnunet-plain", "resenc"}  # Others are fixed or similar.
MEMORY_FRACTIONS = [0.25, 0.5, 0.8]
TRAINING_BATCHES = [2, 4]
INFERENCE_BATCHES = [1, 2, 4]
TRAINING_STEPS = 5
INFERENCE_STEPS = 3
IN_CHANNELS = 4  # As in BraTS, so that inputs and outputs carry realistic weight.
OUT_CHANNELS = 5
MEDIAN_SIZE = [512, 512, 512]

if DRY_RUN:
    NETWORKS = ["nnunet-plain", "nnunet-pocket"]
    MEMORY_FRACTIONS = [0.05]
    TRAINING_BATCHES = [1]
    INFERENCE_BATCHES = [1, 2]
    TRAINING_STEPS = 3
    INFERENCE_STEPS = 2
    MEDIAN_SIZE = [64, 64, 64]

## Measuring

In [ ]:
def clear_memory():
    """Free as much GPU memory as possible, and reset the peak."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def peak_bytes():
    """Return peak allocated and reserved memory since the last reset."""
    if not torch.cuda.is_available():
        return 0.0, 0.0
    return float(torch.cuda.max_memory_allocated()), float(torch.cuda.max_memory_reserved())


def build(name, patch, spacing):
    """Build a network on the GPU."""
    return get_model_from_registry(
        name,
        in_channels=IN_CHANNELS,
        out_channels=OUT_CHANNELS,
        patch_size=list(patch),
        target_spacing=list(spacing),
    ).to(DEVICE)


def training_loss(output):
    """Combine the prediction and any deep supervision outputs, as MIST's loss does."""
    if isinstance(output, dict):
        extras = output.get("deep_supervision") or []
        return output["prediction"].mean() + sum(extra.mean() for extra in extras)
    return output.mean()


def start_steady_peak():
    """Drop what the first step left cached, so later peaks leave it out."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def measure_training(name, patch, spacing, batch, amp):
    """Run real training steps; return first-step and later peaks, and seconds per step."""
    clear_memory()
    model = build(name, patch, spacing).train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    first = (0.0, 0.0)
    start = None
    for step in range(TRAINING_STEPS):
        if step == 2:  # Timed from here: the first step sets up cuDNN and the optimizer,
            # and the second refills the memory that was just freed.
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            start = time.perf_counter()
        optimizer.zero_grad()
        with hardware.autocast_context(amp):
            loss = training_loss(model(torch.randn(batch, IN_CHANNELS, *patch, device=DEVICE)))
        loss.backward()
        optimizer.step()
        if step == 0:
            first = peak_bytes()
            start_steady_peak()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    seconds = (time.perf_counter() - start) / (TRAINING_STEPS - 2) if start else float("nan")
    steady = peak_bytes()
    return (max(first[0], steady[0]), max(first[1], steady[1]), *steady, seconds)


def measure_inference(name, patch, spacing, batch, amp):
    """Run forward passes without gradients; return both peaks and seconds per step."""
    clear_memory()
    model = build(name, patch, spacing).eval()
    with torch.inference_mode(), hardware.autocast_context(amp):
        model(torch.randn(batch, IN_CHANNELS, *patch, device=DEVICE))
        first = peak_bytes()
        start_steady_peak()
        start = time.perf_counter()
        for _ in range(INFERENCE_STEPS - 1):
            model(torch.randn(batch, IN_CHANNELS, *patch, device=DEVICE))
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    seconds = (time.perf_counter() - start) / (INFERENCE_STEPS - 1)
    steady = peak_bytes()
    return (max(first[0], steady[0]), max(first[1], steady[1]), *steady, seconds)

In [ ]:
rows = []


def record(mode, name, spacing_name, patch, batch, precision, estimate, measure, spacing):
    """Run one measurement and save it, keeping going if it runs out of memory."""
    amp = precision == "bf16"
    row = {
        "gpu": GPU_NAME,
        "gpu_memory_gib": round(GPU_MEMORY / GIB, 2),
        "context_gib": round(CONTEXT_BYTES / GIB, 3),
        "mode": mode,
        "network": name,
        "spacing": spacing_name,
        "patch": "x".join(str(p) for p in patch),
        "batch": batch,
        "precision": precision,
        "activations": estimate.activations,
        "parameters": estimate.parameters,
    }
    try:
        allocated, reserved, steady_allocated, steady_reserved, seconds = measure(
            name, patch, spacing, batch, amp
        )
        row.update(
            out_of_memory=False,
            peak_allocated_gib=allocated / GIB,
            peak_reserved_gib=reserved / GIB,
            steady_allocated_gib=steady_allocated / GIB,
            steady_reserved_gib=steady_reserved / GIB,
            seconds_per_step=seconds,
            planner_estimate_gib=pp.estimated_bytes(estimate, batch, amp) / GIB,
        )
        print(
            f"{mode:9} {name:16} {row['patch']:14} batch {batch} {precision}: "
            f"{steady_reserved / GIB:5.1f} GiB (first step {reserved / GIB:5.1f}), {seconds:.2f} s"
        )
    except torch.cuda.OutOfMemoryError:
        row.update(out_of_memory=True)
        print(f"{mode:9} {name:16} {row['patch']:14} batch {batch} {precision}: out of memory")
    rows.append(row)
    pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
    clear_memory()


for name in NETWORKS:
    for spacing_name, spacing in SPACINGS.items():
        if spacing_name == "anisotropic" and name not in ANISOTROPIC_NETWORKS:
            continue
        for precision in PRECISIONS:
            # Plan sizes for this precision, so they suit what it can actually fit.
            patches = []
            for fraction in MEMORY_FRACTIONS:
                plan = pp.plan_patch_and_batch(
                    architecture=name,
                    in_channels=IN_CHANNELS,
                    out_channels=OUT_CHANNELS,
                    median_size=MEDIAN_SIZE,
                    target_spacing=spacing,
                    gpu_memory_bytes=GPU_MEMORY * fraction,
                    amp=precision == "bf16",
                    num_cases=10000,
                )
                if plan.patch_size not in patches:
                    patches.append(plan.patch_size)
            for patch in patches:
                estimate = pp.estimate_network(name, IN_CHANNELS, OUT_CHANNELS, patch, spacing)
                for batch in TRAINING_BATCHES:
                    record(
                        "training",
                        name,
                        spacing_name,
                        patch,
                        batch,
                        precision,
                        estimate,
                        measure_training,
                        spacing,
                    )
                for batch in INFERENCE_BATCHES:
                    record(
                        "inference",
                        name,
                        spacing_name,
                        patch,
                        batch,
                        precision,
                        estimate,
                        measure_inference,
                        spacing,
                    )

results = pd.DataFrame(rows)
print(f"\nDone: {len(results)} measurements, {int(results['out_of_memory'].sum())} out of memory.")

## Results

The ratio is measured peak memory, leaving out the first step, over the planner's
current estimate, for training.
Above 1 means the planner may overfill the GPU for that network; below 1 means it
leaves memory unused. Fitted values are shown so they can be compared by eye, but
the combined CSVs from all four GPUs are what matter.

In [ ]:
measured = results[~results["out_of_memory"]].copy()
measured["total_gib"] = measured["steady_reserved_gib"] + CONTEXT_BYTES / GIB
training = measured[measured["mode"] == "training"].copy()
training["ratio"] = training["total_gib"] / training["planner_estimate_gib"]
print("Training: measured over estimated memory")
print(
    training.pivot_table(
        index="network", columns="precision", values="ratio", aggfunc=["mean", "max"]
    )
    .round(2)
    .to_string()
)

training["first_step_extra_gib"] = training["peak_reserved_gib"] - training["steady_reserved_gib"]
print("\nTraining: extra memory the first step used (GiB)")
print(
    training.pivot_table(
        index="network", columns="precision", values="first_step_extra_gib", aggfunc="max"
    )
    .round(1)
    .to_string()
)


def fit_bytes_per_activation(frame, bytes_per_parameter):
    """Fit peak memory = intercept + slope * activations * batch, parameters removed."""
    x = frame["activations"] * frame["batch"]
    y = frame["steady_reserved_gib"] * GIB - bytes_per_parameter * frame["parameters"]
    if len(frame) < 2 or x.nunique() < 2:
        return float("nan"), float("nan")
    slope, intercept = np.polyfit(x, y, 1)
    return slope, intercept


print("\nFitted bytes per activation (compare with the constants below)")
for mode, per_parameter in (("training", constants.BYTES_PER_PARAMETER), ("inference", 4.0)):
    for precision in PRECISIONS:
        frame = measured[(measured["mode"] == mode) & (measured["precision"] == precision)]
        slope, intercept = fit_bytes_per_activation(frame, per_parameter)
        print(f"  {mode:9} {precision}: {slope:6.2f} bytes, intercept {intercept / GIB:6.2f} GiB")

print("\nCurrent constants")
print(f"  training, FP32:   {constants.BYTES_PER_ACTIVATION}")
print(
    f"  training, BF16:   {constants.BYTES_PER_ACTIVATION * constants.BF16_ACTIVATION_FACTOR:.2f}"
)
print(f"  inference, FP32:  {constants.INFERENCE_BYTES_PER_ACTIVATION}")

## Save the results

Downloads the CSV. Send all three, one per GPU, to fit the constants. If the download
doesn't start, it's in the Files panel on the left.

In [ ]:
print(f"Saved {OUTPUT_CSV}")
try:
    from google.colab import files

    files.download(OUTPUT_CSV)
except ImportError:
    pass